# 07 · Query expansion, diversity, and parent documents

**Level:** advanced · **Time:** 50–65 minutes · **Prerequisites:** notebooks 01–06.

Turn one user request into several targeted searches, implement maximal marginal relevance
(MMR), and expand retrieved child chunks into their parent documents. An optional HyDE
demonstration shows how generated text can guide retrieval without becoming evidence.

**Demo story:** “A two-part question needs coverage, not five copies of the same paragraph.”
Live mode uses a local embedding model and normally makes **two Claude calls** (query
expansion and the final answer). HyDE is an additional explicit opt-in. Offline mode uses
labelled prepared queries and lexical vectors, with no generated answers.


In [ ]:
from pathlib import Path
import sys

# Works when Jupyter starts in the repository or in notebooks/.
ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents]
             if (p / "rag_workshop").is_dir()), None)
if ROOT is None:
    raise RuntimeError("Open this notebook from the LangchainPractical project folder.")
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from rag_workshop import (
    Document, load_documents, chunk_documents, TfidfIndex, DenseIndex,
    bm25_search, reciprocal_rank_fusion, context_text, ask_claude,
    rag_answer, show_hits, settings, offline_mode,
)
import pandas as pd
from IPython.display import display, Markdown

print("Mode:", "OFFLINE: Claude calls skipped" if offline_mode() else "LIVE: Claude calls use API credits")
print("Claude model:", settings().model)


## Step 1 · Establish the baseline

A user may ask about two policies at once. A single similarity query can concentrate results
on one topic. Smaller child chunks let us search precise passages; metadata remembers the
full source document so we can expand context later.


In [ ]:
import json
import numpy as np

documents = load_documents()
chunks = chunk_documents(documents, chunk_size=45, overlap=10)
index = DenseIndex(chunks)
parents = {d.id: d for d in documents}
question = "What are the paid-leave carryover rules, and how often may I work remotely?"
baseline_hits = index.search(question, k=4)
print("Embedding backend:", index.backend)
display(show_hits(baseline_hits))
print("Baseline sources:", sorted({h["metadata"]["source_id"] for h in baseline_hits}))


## Step 2 · Expand the question into search queries

Ask Claude for retrieval queries, not an answer. Preserve the original question so a flawed
rewrite does not replace the user's intent. We parse a JSON list, validate its shape, cap the
list length, and deduplicate. These are validation checks, not a guarantee of faithful rewriting.

**API cost:** one live Claude request. Offline mode uses hand-written baseline queries
explicitly labelled below. They are useful for demonstrating mechanics, not model quality.


In [ ]:
prepared_queries = [
    "Paid leave unused days carryover limit deadline",
    "Remote work days per week manager approval",
]
if offline_mode():
    expanded_queries = prepared_queries
    expansion_origin = "prepared offline baseline (not model-generated)"
else:
    expansion = ask_claude(
        "Return only a JSON array of 2 or 3 concise search queries that together cover "
        "all parts of this question. Preserve policy names and numbers; do not invent facts. "
        "Question: " + question,
        system="You create search queries for a handbook retrieval system.", max_tokens=180,
    )
    raw = expansion["text"].strip()
    if raw.startswith("```"):
        raw = raw.split("\n", 1)[1].rsplit("```", 1)[0].strip()
    try:
        parsed = json.loads(raw)
        if not isinstance(parsed, list) or not all(isinstance(q, str) and q.strip() for q in parsed):
            raise ValueError("Expected a JSON list of nonempty strings")
        expanded_queries = [q.strip()[:250] for q in parsed[:3]]
        expansion_origin = "Claude-generated; inspect for meaning changes"
    except (json.JSONDecodeError, ValueError):
        expanded_queries = prepared_queries
        expansion_origin = "prepared baseline after invalid model output (not model-generated)"
queries = list(dict.fromkeys([question, *expanded_queries]))
print("Expansion source:", expansion_origin)
display(pd.DataFrame({"retrieval_query": queries}))


## Step 3 · Retrieve separately and fuse the lists

Each subquery gets its own top results. RRF combines their rank positions without comparing
raw similarity scores across questions. Expansion adds searches and can add noise; inspect
whether it covers both `benefits_leave` and `remote_work`.


In [ ]:
per_query_hits = [index.search(query, k=4) for query in queries]
expanded_hits = reciprocal_rank_fusion(*per_query_hits, k=8)
for query, hits in zip(queries, per_query_hits):
    print(query)
    print([h["metadata"]["source_id"] for h in hits])
display(show_hits(expanded_hits))
expected_sources = {"benefits_leave", "remote_work"}
for name, hits in (("single query", baseline_hits), ("multi-query", expanded_hits)):
    covered = {h["metadata"]["source_id"] for h in hits}
    print(name, "covers expected sources:", sorted(expected_sources & covered))


## Step 4 · Use MMR to reduce redundant chunks

MMR selects candidates iteratively. The first selection is the most relevant candidate.
Later selections maximize:

`lambda × similarity(query, candidate) − (1 − lambda) × max similarity(candidate, selected)`

With `lambda=1`, selection follows relevance. Lower values penalize redundancy more.
Diversity is not automatically relevance: an unrelated chunk can be very different from
everything already selected. We normalize vectors explicitly for cosine similarity.


In [ ]:
vector_by_id = {d.id: np.asarray(v, dtype=float) for d, v in zip(chunks, index.vectors)}

def unit(vector):
    norm = np.linalg.norm(vector)
    return vector / norm if norm else np.zeros_like(vector)

def mmr_select(query, candidates, k=4, lambda_mult=0.65):
    if not 0 <= lambda_mult <= 1:
        raise ValueError("lambda_mult must be in [0, 1]")
    if not candidates or k <= 0:
        return []
    # Deduplicate candidates before indexing their vectors.
    candidates = list({h["id"]: h for h in candidates}.values())
    matrix = np.stack([unit(vector_by_id[h["id"]]) for h in candidates])
    relevance = matrix @ unit(np.asarray(index.embed_query(query), dtype=float))
    selected, remaining = [], list(range(len(candidates)))
    while remaining and len(selected) < k:
        def objective(i):
            if not selected:
                return float(relevance[i])
            redundancy = float(np.max(matrix[i] @ matrix[selected].T))
            return lambda_mult * float(relevance[i]) - (1 - lambda_mult) * redundancy
        best = max(remaining, key=objective)
        selected.append(best)
        remaining.remove(best)
    return [{**candidates[i], "score": float(relevance[i]),
             "selection": "MMR"} for i in selected]

mmr_hits = mmr_select(question, expanded_hits, k=4, lambda_mult=0.65)
display(show_hits(mmr_hits))


## Step 5 · Compare relevance and diversity

Count distinct sources and inspect the actual passages. A source count is a rough diversity
indicator, not a correctness metric. Our display keeps `score` as query similarity; it is not
the changing MMR objective from the selection loop.


In [ ]:
diversity_rows = []
for weight in (1.0, 0.65, 0.25):
    selected = mmr_select(question, expanded_hits, k=4, lambda_mult=weight)
    sources = {h["metadata"]["source_id"] for h in selected}
    diversity_rows.append({"lambda": weight, "unique_sources": len(sources),
                           "expected_coverage": len(sources & expected_sources),
                           "selected_sources": sorted(sources)})
display(pd.DataFrame(diversity_rows))
print("Read the text: greater variety can also introduce irrelevant evidence.")


## Step 6 · Expand child hits into parent documents

Small chunks make precise retrieval easier but can omit qualifications in neighboring text.
Parent retrieval searches children and returns each matching full source once. This increases
context size, so use a budget and preserve the child IDs that led to each parent.

In production, parent access checks must be at least as strict as child access checks.
This teaching corpus has no per-paragraph access rules.


In [ ]:
def expand_parents(child_hits, max_parents=3):
    grouped = {}
    for hit in child_hits:
        source_id = hit["metadata"]["source_id"]
        if source_id in grouped:
            grouped[source_id]["matched_child_ids"].append(hit["id"])
            continue
        if len(grouped) >= max_parents:
            continue
        parent = parents[source_id]
        grouped[source_id] = {
            "id": parent.id, "text": parent.text,
            "metadata": {**parent.metadata, "source_id": parent.id},
            "score": hit["score"], "matched_child_ids": [hit["id"]],
        }
    return list(grouped.values())

parent_hits = expand_parents(mmr_hits, max_parents=3)
display(pd.DataFrame([
    {"parent_id": hit["id"], "characters": len(hit["text"]),
     "matched_child_ids": hit["matched_child_ids"]} for hit in parent_hits
]))
print("Child context characters:", len(context_text(mmr_hits)))
print("Parent context characters:", len(context_text(parent_hits)))
print(context_text(parent_hits))


## Step 7 · Answer the original question, not a rewritten one

Rewrites guide retrieval; the final answer still responds to the original user request. The
evidence now uses parent document IDs, so those are the citation IDs the model should use.
Expected facts include carryover **up to 5 days**, use by **March 31**, and remote work
**up to 3 days/week with manager approval**.

**API cost:** one live Claude request, subject to the shared response cache. Inspect whether
both required policy sources reached the final evidence; expansion is not a guarantee.


In [ ]:
answer = rag_answer(question, parent_hits, max_tokens=400)
display(Markdown(answer["text"]))
print({key: answer[key] for key in ("input_tokens", "output_tokens", "cached", "offline")})
print("Missing expected sources:", sorted(expected_sources - {h["id"] for h in parent_hits}))


## Step 8 · Optional HyDE: use a hypothesis only for retrieval

Hypothetical Document Embeddings (HyDE) generate a plausible answer passage, embed that
passage, and retrieve real documents near it. The hypothesis may be false. It must never be
included in the final evidence or cited as a source. Its usefulness depends on the task,
embedding model, and evaluation set.

The example is **off by default**. Enabling it in live mode makes one extra Claude request;
offline mode skips it instead of fabricating a model-generated hypothesis.


In [ ]:
RUN_HYDE = False

def retrieve_with_hyde(user_question, k=4):
    if offline_mode():
        return {"status": "skipped offline", "hypothesis": None, "hits": []}
    response = ask_claude(
        "Write a short hypothetical handbook passage that could answer this question. "
        "It is only a retrieval probe, not evidence. Question: " + user_question,
        system="Produce a brief hypothetical passage for a retrieval experiment.",
        max_tokens=150,
    )
    hypothesis = response["text"]
    return {"status": "live", "hypothesis": hypothesis,
            "hits": index.search(hypothesis, k=k)}

if RUN_HYDE:
    hyde_result = retrieve_with_hyde(question)
    print("Status:", hyde_result["status"])
    print("UNVERIFIED RETRIEVAL PROBE:", hyde_result["hypothesis"])
    display(show_hits(hyde_result["hits"]))
    # A later rag_answer(question, hyde_result["hits"]) would use only real corpus text.
else:
    print("Optional HyDE experiment skipped.")


## Your turn

1. Compare single-query and multi-query retrieval for Team price plus retention duration.
2. Use hand-written subqueries first, then compare Claude rewrites if you choose to pay for them.
3. Expand parents and check whether complete qualifications are preserved.
4. Explain a failure case for each technique: rewriting, MMR, parent expansion, and HyDE.

**Presenter checkpoint:** changing more than one retrieval technique at once makes results
hard to explain. Show the baseline, then add one technique and inspect the changed evidence.


In [ ]:
# Sample solution: no extra model request is needed for this comparison.
exercise_question = "What does Team cost, and how long is its data retained?"
exercise_queries = [exercise_question, "Team price annual billing", "Team data retention days"]
exercise_rankings = [index.search(q, k=4) for q in exercise_queries]
exercise_fused = reciprocal_rank_fusion(*exercise_rankings, k=8)
exercise_selected = mmr_select(exercise_question, exercise_fused, k=4, lambda_mult=0.75)
exercise_parents = expand_parents(exercise_selected, max_parents=3)
display(show_hits(exercise_parents))
print("Rewriting can alter intent; MMR can select irrelevant variety;")
print("parent expansion can add distracting text; HyDE can steer retrieval toward false assumptions.")


## What you can now demonstrate

Multi-query retrieval, rank fusion, explicit MMR selection, and parent-context expansion.
Measure these techniques against a representative labelled set before deciding to deploy them.
Query expansion adds model latency and search work; parent expansion adds prompt tokens.
Offline execution verifies mechanics, while semantic-quality claims require real embeddings.

**Next:** evaluate retrieval and answer quality, then harden the pipeline in notebooks 08–10.

Primary references: [HyDE paper](https://arxiv.org/abs/2212.10496),
[MMR paper](https://www.cs.cmu.edu/~jgc/publication/The_Use_MMR_Diversity_Based_LTMIR_1998.pdf),
[Sentence Transformers semantic search](https://www.sbert.net/examples/sentence_transformer/applications/semantic-search/README.html).
